# Market Foundation Model — Google Colab (T4)
Notebook resmi untuk menjalankan pipeline dari data mentah sampai fine-tuning. Jalankan berurutan. Runtime GPU T4 direkomendasikan; data dan checkpoint disimpan di Google Drive agar tidak hilang saat runtime terputus.

**Protokol:** tokenizer/pretraining/fine-tuning 2015–2020, pemilihan checkpoint 2021, kalibrasi 2022, final test 2023. Jangan mengubah split setelah melihat hasil final test.

> Catatan: training tidak menjamin profit. Periksa kualitas data, missing years, biaya transaksi, dan hasil out-of-sample.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/content/market-foundation-model')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/NodePro-gif/market-foundation-model.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
%cd /content/market-foundation-model
print('Repository ready:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


## 1. GPU dan dependensi
Pilih **Runtime → Change runtime type → T4 GPU** sebelum menjalankan sel ini.

In [ ]:
# Colab provides the CUDA-enabled PyTorch build; don't reinstall torch here.
!pip -q install "numpy>=1.24,<3" "pandas>=2,<4" "scikit-learn>=1.3,<2" "matplotlib>=3.7,<4"

import torch, os
if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak aktif. Pilih runtime T4 GPU lalu jalankan ulang.')
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GiB')
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('AMP: FP16 | SDPA attention: PyTorch automatic kernel selection')


## 2. Hubungkan Google Drive
Data diproses pada disk lokal Colab (`/content`) agar I/O cepat. Artefak dipulihkan dari dan disinkronkan kembali ke `MyDrive/market-foundation-model`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil

drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/market-foundation-model')
STORE.mkdir(parents=True, exist_ok=True)

# Train/read on the Colab VM disk, not through the slower Drive FUSE mount.
# Restore previous artifacts once at startup; sync important outputs back after stages.
for name in ['corpus', 'npz', 'mkt', 'ckpt', 'ftune']:
    local = Path('/content') / name
    persistent = STORE / name
    persistent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink() or local.is_file():
        local.unlink()
    elif local.exists() and not local.is_dir():
        local.unlink()
    local.mkdir(parents=True, exist_ok=True)
    shutil.copytree(persistent, local, dirs_exist_ok=True)

def sync_to_drive(name):
    src, dst = Path('/content') / name, STORE / name
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Synced {name} to Drive')

os.environ['MFM_MKT_DIR'] = '/content/mkt'
os.environ['MFM_CKPT_DIR'] = '/content/ckpt'
os.environ['MFM_CKPT_BACKUP_DIR'] = str(STORE / 'ckpt')
os.environ['MFM_FT_DIR'] = '/content/ftune'
print('Fast local workspace: /content | Persistent backup:', STORE)


## 3. Unduh dan verifikasi data
Tahap ini mengunduh OHLCV M5 untuk seluruh aset dan M1 untuk XAUUSD/EURUSD dari penyedia data yang dikonfigurasi di `stage1_corpus.py`. File yang tidak tersedia akan dilaporkan; periksa manifest sebelum melanjutkan.

In [ ]:
%cd /content/market-foundation-model
!python -u stage1_corpus.py
sync_to_drive('corpus')
sync_to_drive('npz')


## 4. Bangun tokenizer dan label
Tahap ini CPU/RAM-intensive dan dapat memerlukan waktu. Statistik kuantil, regime, dan BPE hanya dipelajari dari periode training 2015–2020 dan aset non-held-out.

In [ ]:
!python -u test_tokenizer.py
!python -u stage2_tokenizer.py
!ls -lh /content/mkt | head -30
sync_to_drive('mkt')


## 5. Pretraining GPT
Default T4: micro-batch 64 × accumulation 1 (effective batch 64), FP16 AMP, 6.000 optimizer steps. DataLoader memakai 4 worker dan prefetch 4; token NPZ dicache sebagai array terdekompresi per worker. `resume.pt` disimpan berkala. Jika runtime terputus, jalankan ulang sel ini dengan `PRETRAIN_RESUME=1`; resume memulihkan bobot dan optimizer, tetapi DataLoader memulai ulang epoch berjalan sehingga beberapa batch dapat terulang. Jika tokenizer dibuat ulang, hapus `ckpt/resume.pt` terlebih dahulu karena checkpoint lama terikat ke fingerprint tokenizer sebelumnya.

In [ ]:
import os
# T4 throughput profile: larger micro-batch, same effective batch as before.
os.environ['PRETRAIN_BATCH'] = '64'
os.environ['PRETRAIN_ACCUM'] = '1'
os.environ['PRETRAIN_STEPS'] = '6000'
os.environ['PRETRAIN_CKPT_EVERY'] = '1000'
os.environ['PRETRAIN_EVAL_EVERY'] = '1000'
os.environ['PRETRAIN_WORKERS'] = '4'
os.environ['PRETRAIN_PREFETCH'] = '4'
os.environ['MFM_TORCH_COMPILE'] = '1'
os.environ['MFM_COMPILE_MODE'] = 'reduce-overhead'
os.environ['PRETRAIN_RESUME'] = '1'
!python -u stage3_pretrain.py
sync_to_drive('ckpt')


## 6. Verifikasi checkpoint dan tokenizer
Pemeriksaan ini menggunakan **validasi 2021**, bukan final test 2023. Fine-tuning akan menolak checkpoint tanpa manifest tokenizer yang cocok.

In [ ]:
!python -u verify_ckpt.py 20


## 7. Fine-tuning multi-task (multi-asset)
Shared heads are trained on balanced samples from all available non-held-out assets (2015–2020); XAUUSD 2021 selects the checkpoint. Default T4: micro-batch 64 × accumulation 4 (effective batch 256), AMP FP16, 2 worker dan prefetch 4. Checkpoint dipilih hanya menggunakan AUC validasi 2021. Dataset final test 2023 baru dibuat setelah checkpoint terkunci.

In [ ]:
import os
# T4 throughput profile: effective batch stays at 256 while GPU micro-batch doubles.
os.environ['ST4_EPOCHS'] = '3'
os.environ['ST4_BATCH'] = '64'
os.environ['ST4_ACCUM'] = '4'
os.environ['ST4_LR'] = '1e-4'
os.environ['ST4_WARMUP_RATIO'] = '0.08'
os.environ['ST4_MIN_LR_RATIO'] = '0.10'
os.environ['ST4_PATIENCE'] = '3'
os.environ['ST4_STRIDE'] = '32'
os.environ['ST4_VAL_STRIDE'] = '4'
os.environ['ST4_TEST_STRIDE'] = '4'
os.environ['ST4_HELD_STRIDE'] = '8'
os.environ['ST4_AMP'] = '1'
os.environ['ST4_WORKERS'] = '4'
os.environ['ST4_PREFETCH'] = '4'
os.environ['MFM_TORCH_COMPILE'] = '1'
os.environ['MFM_COMPILE_MODE'] = 'reduce-overhead'
!python -u stage4_finetune.py
sync_to_drive('ftune')


## 8. Kalibrasi dan backtest
Jalankan kalibrasi hanya pada 2022, lalu backtest pada 2023 dengan biaya transaksi realistis. Pastikan skrip evaluasi menggunakan checkpoint baru dari sesi ini.

In [ ]:
!python -u validate_stage4.py
!python -u backtest_stage4.py
sync_to_drive('ftune')
sync_to_drive('mkt')
sync_to_drive('corpus')
sync_to_drive('npz')


## Lokasi artefak
- Data mentah: `MyDrive/market-foundation-model/corpus`
- Data hasil parsing: `npz/`
- Tokenizer dan label: `mkt/`
- Checkpoint dan resume: `ckpt/`
- Hasil fine-tuning: `ftune/`

Simpan commit SHA, konfigurasi, versi GPU/CUDA, jumlah data, dan asumsi spread/komisi/slippage bersama setiap hasil eksperimen.